In [2]:
!pip install -qU langchain langchain-google-genai tavily-python requests

In [11]:
import os
from google.colab import userdata

GEMINI_API_KEY      = userdata.get("GEMINI_API_KEY")
WEATHER_API_KEY = userdata.get("WEATHER_API_KEY")
TAVILY_API_KEY      = userdata.get("TAVILY")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["TAVILY"] = TAVILY_API_KEY

print("API keys loaded from Colab Secrets!")

API keys loaded from Colab Secrets!


In [25]:
CITY        = "Tokyo"                              # Destination city
DURATION    = 5                                    # Number of days
INTERESTS   = ["food", "culture", "temples"]       # Your interests
TIME_OF_DAY = "morning"                            # morning / afternoon / evening / full day

print(f" Destination  : {CITY}")
print(f" Duration     : {DURATION} days")
print(f" Interests    : {', '.join(INTERESTS)}")
print(f" Time of Day  : {TIME_OF_DAY}")

 Destination  : Tokyo
 Duration     : 5 days
 Interests    : food, culture, temples
 Time of Day  : morning


In [26]:
import requests

def get_weather(city: str, api_key: str):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    temp_c    = round(data["main"]["temp"] - 273.15, 1)
    feels_c   = round(data["main"]["feels_like"] - 273.15, 1)
    humidity  = data["main"]["humidity"]
    condition = data["weather"][0]["description"].capitalize()
    wind      = data["wind"]["speed"]
    city_name = data["name"]
    country   = data["sys"]["country"]

    summary = (
        f" Location    : {city_name}, {country}\n"
        f"  Temperature : {temp_c}°C (feels like {feels_c}°C)\n"
        f"  Condition   : {condition}\n"
        f" Humidity    : {humidity}%\n"
        f" Wind Speed  : {wind} m/s"
    )
    return summary

weather_summary = get_weather(CITY, WEATHER_API_KEY)

print("\n Current Weather")
print("=" * 40)
print(weather_summary)


 Current Weather
 Location    : Tokyo, JP
  Temperature : 7.3°C (feels like 2.8°C)
  Condition   : Light intensity shower rain
 Humidity    : 89%
 Wind Speed  : 9.26 m/s


In [27]:
from tavily import TavilyClient

client = TavilyClient(api_key=TAVILY_API_KEY)

def tavily_search(query: str, max_results: int = 5) -> str:
    results = client.search(query=query, max_results=max_results)
    output = []
    for r in results.get("results", []):
        title   = r.get("title", "No title")
        content = r.get("content", "")[:300].strip()
        url     = r.get("url", "")
        output.append(f"• {title}\n  {content}\n  🔗 {url}")
    return "\n\n".join(output) if output else "No results found."

interests_str = ", ".join(INTERESTS)

print("🔍 Fetching live recommendations...\n")

hotels_raw      = tavily_search(f"best hotels to stay in {CITY} for tourists")
restaurants_raw = tavily_search(f"top restaurants in {CITY} for {interests_str}")
attractions_raw = tavily_search(f"top attractions and things to do in {CITY} for {interests_str}")

print(" Hotels:\n",      hotels_raw[:600],      "\n")
print(" Restaurants:\n", restaurants_raw[:600], "\n")
print(" Attractions:\n", attractions_raw[:600])

🔍 Fetching live recommendations...

 Hotels:
 • Where to Stay in Tokyo [7 Best Areas in Tokyo for Tourists]
  Hotel Sunroute Plaza Shinjuku (3.5*) – Best for practical travelers. The hotel sits on a fantastic location in Shinjuku with reasonable price.
  🔗 https://asiatravelbug.com/blog/where-to-stay-in-tokyo-first-time-best-area-family/

• Where to Stay in Tokyo for the Best First Time Experience
  Whenever anyone is on a first time visit to Tokyo I tell them to stay in Shibuya. In fact, when I took my friend Danielle to Tokyo for the first time, Shibuya was the first place we stayed! While I would definitely try to visit Harajuku while  

 Restaurants:
 • Best Food & Temple Spots in Tokyo: 5 Must-Try Gems - TravelBubu
  This guide covers 5 spots in Tokyo, including Beef Ramen Shop (Tokyo), Pride Fish Wholesale, Sensoji Temple (Asakusa), Unagi Restaurant near
  🔗 https://travelbubu.com/guides/best-food-temple-spots-in-tokyo-5-must-try-gems/

• Top 5 Tokyo Restaurants Where You Can Exp

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    google_api_key=GEMINI_API_KEY
)

prompt_template = ChatPromptTemplate.from_template("""
You are an expert travel planner. Generate a detailed, personalized travel plan using the real-time data below.

=== TRIP DETAILS ===
City         : {city}
Duration     : {duration} days
Interests    : {interests}
Preferred Time of Day : {time_of_day}

=== REAL-TIME WEATHER ===
{weather_summary}

=== HOTELS (Live Data) ===
{hotels}

=== RESTAURANTS (Live Data) ===
{restaurants}

=== ATTRACTIONS (Live Data) ===
{attractions}

=== INSTRUCTIONS ===
Generate the travel plan with exactly these sections:

## 🌤️ Weather Summary & Packing Tips
Summarize the weather and advise what to wear or carry.

## 🏛️ Recommended Attractions
List 5–7 attractions tailored to the traveler's interests ({interests}).
For each: name, why visit, best time, how long to spend.

## 🏨 Recommended Hotels
List 3–4 hotels with price range (budget / mid-range / luxury) and highlights.

## 🍽️ Recommended Restaurants
List 5–6 restaurants with cuisine type, must-try dish, and price range.

## 📅 Day-by-Day Itinerary ({duration} Days)
Create a detailed plan for each day.
- Structure activities around the preferred time: {time_of_day}
- Each day should include: Morning / Afternoon / Evening
- Include meals, attractions, transit tips, and local advice

## 💡 Local Tips & Insider Advice
Hidden gems, cultural etiquette, transport hacks, money-saving tips.

Be specific, vivid, and practical. Use emojis to improve readability.
""")

chain = prompt_template | llm | StrOutputParser()

print("LangChain + Gemini pipeline ready!")

LangChain + Gemini pipeline ready!


In [23]:
from IPython.display import Markdown, display

print(f"⏳ Generating your {DURATION}-day travel plan for {CITY}...\n")

travel_plan = chain.invoke({
    "city"            : CITY,
    "duration"        : DURATION,
    "interests"       : ", ".join(INTERESTS),
    "time_of_day"     : TIME_OF_DAY,
    "weather_summary" : weather_summary,
    "hotels"          : hotels_raw,
    "restaurants"     : restaurants_raw,
    "attractions"     : attractions_raw,
})

print("=" * 60)
display(Markdown(travel_plan))

⏳ Generating your 5-day travel plan for Tokyo...



Konnichiwa! Welcome to Tokyo! As your expert travel planner, I've crafted a personalized 5-day itinerary based on your interests in food, culture, and temples, prioritizing morning activities, and taking into account the current weather conditions. Get ready for an unforgettable journey through Japan's vibrant capital!

---

## 🌤️ Weather Summary & Packing Tips

Brace yourself for a chilly and damp start to your Tokyo adventure!
*   **Temperature:** A brisk 7.3°C, but with the wind chill, it will feel more like a freezing 2.8°C.
*   **Condition:** Expect light intensity shower rain throughout your stay.
*   **Humidity:** High at 89%, which can make the cold feel even more penetrating.
*   **Wind Speed:** A strong 9.26 m/s, so wind protection is crucial.

**Packing Tips:**
*   **Layers are Key:** Pack thermal base layers, a warm sweater or fleece, and a substantial waterproof and windproof outer jacket.
*   **Waterproof Everything:** A sturdy umbrella is a must-have. Consider waterproof pants if you plan to spend extended time outdoors.
*   **Warm Footwear:** Comfortable, waterproof walking shoes or boots are essential, as you'll be doing a lot of walking in potentially wet conditions.
*   **Accessories:** Don't forget a warm hat, gloves, and a scarf to protect against the wind and cold.
*   **Small Backpack:** A small, waterproof daypack will be handy for carrying your umbrella, water bottle, and any souvenirs.

---

## 🏛️ Recommended Attractions

Here are 5-7 attractions perfectly aligned with your interests in food, culture, and temples:

1.  **Sensoji Temple (Asakusa) ⛩️**
    *   **Why Visit:** Tokyo's oldest temple, offering a deep dive into traditional Japanese culture. The Nakamise-dori market leading up to the temple is a fantastic place for traditional snacks and souvenirs, blending food and culture seamlessly.
    *   **Best Time:** Early morning (before 9 AM) to avoid crowds and experience a serene atmosphere, especially given your preferred time of day.
    *   **How Long to Spend:** 2-3 hours.

2.  **Meiji Jingu Shrine (Shibuya/Harajuku) 🌳**
    *   **Why Visit:** A peaceful oasis dedicated to Emperor Meiji and Empress Shoken. The vast forested grounds offer a tranquil escape and a profound cultural experience, contrasting beautifully with the nearby bustling districts.
    *   **Best Time:** Morning, shortly after opening (around 6 AM, check seasonal times) for a contemplative walk.
    *   **How Long to Spend:** 1.5-2 hours.

3.  **Asakusa Neighborhood (Food & Culture Hub) 🏮**
    *   **Why Visit:** Beyond Sensoji Temple, Asakusa is a living museum of old Tokyo. Explore traditional shops, try local street food (like melon pan or age-manju), and soak in the nostalgic atmosphere. You can also experience traditional cultural activities like rickshaw rides.
    *   **Best Time:** Morning to early afternoon, after visiting Sensoji, to enjoy the food stalls and shops.
    *   **How Long to Spend:** 3-4 hours (including Sensoji).

4.  **Shinjuku Gyoen National Garden (Shinjuku) 🌸**
    *   **Why Visit:** A magnificent urban oasis featuring three distinct garden styles (Japanese Traditional, French Formal, and English Landscape). It’s a perfect spot to appreciate Japanese aesthetics and find tranquility amidst the city's hustle.
    *   **Best Time:** Morning, right at opening, for a quiet stroll. The rain will enhance the lush greenery.
    *   **How Long to Spend:** 2-2.5 hours.

5.  **Tsukiji Outer Market (Tsukiji) 🍣**
    *   **Why Visit:** While the famous inner fish market moved to Toyosu, the Tsukiji Outer Market remains a vibrant hub for fresh seafood, produce, kitchenware, and incredible street food. It's a fantastic place to immerse yourself in Tokyo's culinary culture.
    *   **Best Time:** Early morning (7-9 AM) for the freshest offerings and bustling atmosphere.
    *   **How Long to Spend:** 2-3 hours.

6.  **Tokyo National Museum (Ueno Park) 🖼️**
    *   **Why Visit:** Japan's largest and oldest national museum, housing an extensive collection of Japanese and Asian art and archaeological artifacts. It's an excellent indoor option for cultural immersion, especially on a rainy day.
    *   **Best Time:** Morning, right after opening, to enjoy the exhibits without large crowds.
    *   **How Long to Spend:** 3-4 hours.

---

## 🏨 Recommended Hotels

Here are 3-4 hotel recommendations catering to different preferences and budgets, based on live data:

1.  **Hotel Sunroute Plaza Shinjuku (Shinjuku) 🏨**
    *   **Price Range:** Mid-range
    *   **Highlights:** "Best for practical travelers." Excellent location in Shinjuku, offering great access to transportation, dining, and nightlife. A reliable choice for comfort and convenience at a reasonable price.

2.  **Park Hotel Tokyo (Minato) ✨**
    *   **Price Range:** Luxury
    *   **Highlights:** "Stunning panoramic views over train lines, Tokyo Tower, Zenko-ji Temple, and even Mount Fuji on a clear day." Located in Minato, known for its artistic design and exceptional service. Perfect for those seeking a sophisticated experience with breathtaking vistas.

3.  **Shibuya Area Hotel (e.g., Shibuya Excel Hotel Tokyu) 🏙️**
    *   **Price Range:** Mid-range to Luxury (depending on specific hotel)
    *   **Highlights:** Recommended for a "first-time visit to Tokyo." Staying in Shibuya puts you right in the heart of the action, with iconic Shibuya Crossing, trendy shops, and diverse dining options at your doorstep. Offers a vibrant, immersive Tokyo experience.

---

## 🍽️ Recommended Restaurants

Indulge in Tokyo's culinary delights with these 5-6 recommendations, focusing on your interests:

1.  **Beef Ramen Shop (Tokyo) 🍜**
    *   **Cuisine Type:** Ramen
    *   **Must-Try Dish:** Signature Beef Ramen (specific dish may vary by actual shop, but look for a rich broth and tender beef slices).
    *   **Price Range:** Mid-range
    *   **Why Visit:** Experience a local favorite, perfect for warming up on a chilly, rainy day.

2.  **Unagi Restaurant near Sensoji Temple (Asakusa) 🍣**
    *   **Cuisine Type:** Unagi (grilled eel)
    *   **Must-Try Dish:** Unadon (grilled eel over rice) or Unaju (eel served in a lacquered box).
    *   **Price Range:** Mid-range to Luxury
    *   **Why Visit:** A traditional and highly sought-after Japanese delicacy, especially authentic in the historic Asakusa area.

3.  **Yakatabune Amiko (Sumida River) 🛥️**
    *   **Cuisine Type:** Traditional Japanese Cuisine (Kaiseki-style)
    *   **Must-Try Dish:** Enjoy a multi-course dinner with fresh seafood and seasonal ingredients while cruising the Sumida River.
    *   **Price Range:** Luxury
    *   **Why Visit:** Offers a unique "food and traditional culture" experience on a traditional houseboat, providing stunning views of Tokyo's skyline.

4.  **Oku-Akasaka Sushi Tanji (Akasaka) 🍣**
    *   **Cuisine Type:** Sushi
    *   **Must-Try Dish:** Omakase (chef's choice) for a curated, high-quality sushi experience.
    *   **Price Range:** Luxury
    *   **Why Visit:** For an authentic and refined sushi experience, often considered an art form in Japan.

5.  **Chanko Sakaba Edosawa (Ryogoku) 🍲**
    *   **Cuisine Type:** Chanko Nabe (sumo wrestler's hot pot)
    *   **Must-Try Dish:** Various Chanko Nabe pots, often with chicken, seafood, and vegetables.
    *   **Price Range:** Mid-range
    *   **Why Visit:** A hearty and communal dining experience, deeply rooted in Japanese culture, especially in the sumo district of Ryogoku. Perfect for cold weather.

6.  **Pride Fish Wholesale (Tsukiji Outer Market) 🐟**
    *   **Cuisine Type:** Fresh Seafood (sashimi, grilled, etc.)
    *   **Must-Try Dish:** Fresh sashimi platters or grilled seafood skewers.
    *   **Price Range:** Budget to Mid-range
    *   **Why Visit:** Experience the freshest seafood directly from the market, a quintessential Tokyo food experience.

---

## 📅 Day-by-Day Itinerary (5 Days)

This itinerary prioritizes your morning preference and incorporates your interests, along with practical tips for the current weather.

### Day 1: Arrival & Shinjuku's Serenity to City Lights

*   **Morning (Preferred Time):**
    *   **Arrival & Check-in:** Arrive at Narita (NRT) or Haneda (HND) Airport. Take the Narita Express (N'EX) or Keikyu Line to your hotel (e.g., Hotel Sunroute Plaza Shinjuku). Check in and drop off luggage.
    *   **Activity: Shinjuku Gyoen National Garden 🌸 (2.5 hours)**
        *   Head straight for this tranquil oasis. The rain will make the gardens feel even more serene and vibrant. Wear your waterproofs and enjoy a peaceful morning stroll through the different garden styles.
    *   **Transit Tip:** From Shinjuku Station, it's a 10-15 minute walk to the garden's Shinjuku Gate.
*   **Afternoon:**
    *   **Lunch: Local Shinjuku Eatery 🍜**
        *   Explore the side streets of Shinjuku for a cozy ramen shop or a casual Japanese diner. Look for a "Beef Ramen Shop" recommendation from your list if you find one nearby.
    *   **Activity: Explore Shinjuku (2-3 hours)**
        *   Wander through the bustling streets of Shinjuku, perhaps visiting the Tokyo Metropolitan Government Building for free panoramic city views (an excellent indoor activity on a rainy day).
*   **Evening:**
    *   **Dinner: Izakaya Experience 🏮**
        *   Enjoy a traditional Japanese izakaya (pub) experience in Shinjuku Golden Gai or Omoide Yokocho (Piss Alley) for grilled skewers and drinks.
    *   **Local Advice:** Golden Gai is famous for its tiny, unique bars. Omoide Yokocho offers a nostalgic atmosphere with delicious yakitori.

### Day 2: Ancient Temples & Asakusa Delights

*   **Morning (Preferred Time):**
    *   **Activity: Sensoji Temple & Nakamise-dori Market ⛩️ (3 hours)**
        *   Start your day early at Sensoji Temple in Asakusa. Arriving before 9 AM will allow you to experience the temple's grandeur and the Nakamise-dori market's traditional charm with fewer crowds. Enjoy the vibrant stalls even in the rain.
    *   **Local Advice:** Purchase some traditional snacks like *kaminari okoshi* (rice crackers) or *age-manju* (fried bean buns) from Nakamise-dori.
*   **Afternoon:**
    *   **Lunch: Unagi Restaurant near Sensoji Temple 🍣**
        *   Indulge in a traditional unagi (grilled eel) meal at one of the renowned restaurants in Asakusa. A perfect, warming dish for a cool day.
    *   **Activity: Explore Asakusa & Sumida River Cruise (2-3 hours)**
        *   Continue exploring the Asakusa neighborhood. Consider a short Sumida River cruise (covered boat) for unique views of Tokyo, including the Skytree, offering a comfortable way to see sights despite the rain.
    *   **Transit Tip:** Asakusa is easily accessible via the Ginza Subway Line or Asakusa Subway Line.
*   **Evening:**
    *   **Dinner: Chanko Sakaba Edosawa 🍲 (Ryogoku)**
        *   Head to the Ryogoku area (home of sumo wrestling) for a hearty Chanko Nabe (sumo wrestler's hot pot). It’s an authentic cultural experience and wonderfully warming.
    *   **Local Advice:** Chanko Nabe is a communal dish, great for sharing.

### Day 3: Serene Shrines, Youth Culture & Shibuya Buzz

*   **Morning (Preferred Time):**
    *   **Activity: Meiji Jingu Shrine 🌳 (2 hours)**
        *   Begin your day with a visit to the tranquil Meiji Jingu Shrine. Walk through the giant torii gate and along the peaceful forested path. The morning mist or light rain will add to its spiritual atmosphere.
    *   **Transit Tip:** The shrine is a short walk from Harajuku Station or Meiji-jingumae Station.
*   **Afternoon:**
    *   **Lunch: Harajuku Cafe/Crepes 🥞**
        *   Explore the vibrant and quirky Takeshita Street in Harajuku. Grab a trendy crepe or a casual lunch at one of the many cafes.
    *   **Activity: Harajuku & Shibuya Crossing (3-4 hours)**
        *   Immerse yourself in Harajuku's unique youth fashion and culture. Then, take a short train ride to Shibuya to witness the iconic Shibuya Crossing. Explore the surrounding shops and department stores.
*   **Evening:**
    *   **Dinner: Oku-Akasaka Sushi Tanji (Akasaka) 🍣**
        *   Treat yourself to a high-end sushi experience in Akasaka. An omakase (chef's choice) menu will provide a memorable culinary journey.
    *   **Local Advice:** Book sushi restaurants in advance, especially for omakase. Dress smart casual.

### Day 4: Market Flavors & Cultural Immersion

*   **Morning (Preferred Time):**
    *   **Activity: Tsukiji Outer Market 🍣 (2.5-3 hours)**
        *   Get an early start to experience the bustling Tsukiji Outer Market. Sample fresh seafood, try tamagoyaki (rolled omelet), and explore the various food stalls. This is a prime spot for food interest.
    *   **Lunch: Pride Fish Wholesale (Tsukiji Outer Market) 🐟**
        *   Enjoy the freshest sashimi bowls or grilled fish right in the market.
    *   **Transit Tip:** Tsukiji is accessible via the Hibiya Subway Line (Tsukiji Station) or Oedo Subway Line (Tsukijishijo Station).
*   **Afternoon:**
    *   **Activity: Tokyo National Museum (Ueno Park) 🖼️ (3-4 hours)**
        *   Head to Ueno Park and spend the afternoon at the Tokyo National Museum. It's an excellent indoor activity to escape the rain and dive deep into Japanese art and history. Ueno Park also has other museums and a zoo if you have extra time.
*   **Evening:**
    *   **Dinner: Yakatabune Amiko Cruise 🛥️ (Sumida River)**
        *   Experience a traditional Yakatabune dinner cruise on the Sumida River. Enjoy exquisite Japanese cuisine while taking in the illuminated Tokyo skyline from the comfort of a covered boat.
    *   **Local Advice:** This is a fantastic way to combine food, culture, and sightseeing, especially appealing on a rainy evening. Book in advance.

### Day 5: Last Bites & Departure

*   **Morning (Preferred Time):**
    *   **Activity: Last-Minute Souvenir Shopping / Ginza Stroll 🛍️ (2-3 hours)**
        *   Depending on your flight schedule, enjoy some last-minute souvenir shopping around your hotel or explore the upscale Ginza district. Ginza offers department stores, boutiques, and cafes.
    *   **Brunch/Lunch:**
        *   Have a final delicious Japanese meal at a local cafe or restaurant in Ginza, perhaps trying a traditional Japanese breakfast set or a light lunch.
*   **Afternoon:**
    *   **Departure:** Head back to Narita (NRT) or Haneda (HND) Airport for your departure, carrying wonderful memories of Tokyo.
    *   **Transit Tip:** Plan your airport transfer well in advance, considering potential traffic or train delays.

---

## 💡 Local Tips & Insider Advice

*   **Transport Hacks: Tokyo Metro & JR Pass 🚇**
    *   Get a **Suica or Pasmo IC card** immediately upon arrival. These rechargeable cards work on all trains and buses in Tokyo (and most of Japan), saving you time from buying individual tickets.
    *   Utilize **Google Maps** for real-time train schedules and routes. It's incredibly accurate.
    *   Consider a **Tokyo Subway Ticket** (24/48/72-hour pass) if you plan heavy subway use within a short period, but for 5 days with varied destinations, an IC card might be more flexible.
*   **Cultural Etiquette 🙏**
    *   **Bow:** A slight bow is a common greeting and sign of respect.
    *   **Remove Shoes:** Always remove your shoes when entering homes, temples, traditional restaurants, and some changing rooms. Look for shoe lockers or racks.
    *   **Chopsticks:** Don't stick chopsticks upright in your rice (it resembles a funeral ritual). Don't pass food from chopstick to chopstick. Use the back of your chopsticks or serving utensils for communal dishes.
    *   **Quiet on Public Transport:** Keep conversations low and avoid talking on your phone on trains and buses.
    *   **Tipping:** Tipping is not customary in Japan and can sometimes be seen as rude. Excellent service is standard.
*   **Money-Saving Tips 💰**
    *   **Convenience Stores:** Embrace Konbini (7-Eleven, FamilyMart, Lawson) for cheap and delicious meals, snacks, and drinks. They also have ATMs that accept international cards.
    *   **Lunch Sets:** Many restaurants offer excellent value "lunch sets" (teishoku) that are significantly cheaper than dinner.
    *   **Water:** Carry a reusable water bottle. Many hotels and public places have water fountains.
    *   **Free Views:** Utilize free observation decks like the Tokyo Metropolitan Government Building instead of paying for Skytree/Tokyo Tower (unless you specifically want that experience).
*   **Hidden Gems & Rainy Day Alternatives 🌧️**
    *   **Ghibli Museum (Mitaka):** If you're a Studio Ghibli fan, this is a magical experience. *Book tickets months in advance, as they sell out fast.*
    *   **teamLab Planets TOKYO (Toyosu):** An immersive digital art museum, perfect for an indoor, unique experience.
    *   **Capsule Hotels:** Consider trying a night in a capsule hotel for a unique, budget-friendly experience.
    *   **Depachika (Department Store Food Basements):** Explore the incredible food halls in the basaki (basement) of major department stores (like Isetan, Takashimaya). They offer gourmet foods, bentos, and desserts – great for a quick, high-quality meal or snack.

Enjoy your incredible journey through Tokyo!

In [29]:
filename = f"{CITY.replace(' ', '_')}_{DURATION}day_travel_plan.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write(f"# ✈️ {CITY} — {DURATION}-Day Travel Plan\n\n")
    f.write(f"**Interests:** {', '.join(INTERESTS)}  \n")
    f.write(f"**Preferred Time:** {TIME_OF_DAY}  \n\n")
    f.write("---\n\n")
    f.write(travel_plan)

print(f"Saved to: {filename}")

from google.colab import files
files.download(filename)

Saved to: Tokyo_5day_travel_plan.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>